In [11]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By

In [12]:
driver = webdriver.Chrome()

url = "https://ev.or.kr/nportal/partcptn/initFaqAction.do"
driver.get(url)

time.sleep(2)

In [13]:
def extract_faq_on_current_page(driver):
    data = []

    faq_items = driver.find_elements(
        By.CSS_SELECTOR,
        "#subPage > div > div > div.contentList > div"
    )

    for item in faq_items:
        question = ""
        answer = ""

        try:
            question = item.find_element(
                By.CSS_SELECTOR,
                "div.faq_title > div.title"
            ).text.strip()
        except:
            pass

        try:
            answer = item.find_element(
                By.CSS_SELECTOR,
                "div.faq_con"
            ).text.strip()
        except:
            pass

        if question and answer:
            data.append({
                "question": question,
                "answer": answer
            })

    return data

In [14]:
def go_to_next_page(driver, next_page):
    target = str(next_page)

    # 1차: 숫자 링크 직접 클릭
    links = driver.find_elements(By.XPATH, f"//a[normalize-space()='{target}']")
    if links:
        try:
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", links[0])
            time.sleep(0.5)
            driver.execute_script("arguments[0].click();", links[0])
            time.sleep(2)
            return True
        except:
            pass

    # 2차: onclick이나 javascript href 안에 숫자가 있는 경우
    links = driver.find_elements(By.CSS_SELECTOR, "a")
    for link in links:
        try:
            txt = link.text.strip()
            href = link.get_attribute("href") or ""
            onclick = link.get_attribute("onclick") or ""

            if txt == target or f"'{target}'" in onclick or f"({target}" in onclick or target in href:
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", link)
                time.sleep(0.5)
                driver.execute_script("arguments[0].click();", link)
                time.sleep(2)
                return True
        except:
            pass

    return False

In [15]:
all_data = []

start_page = 1
end_page = 4

for page in range(start_page, end_page + 1):
    print(f"{page}페이지 수집 중...")

    page_data = extract_faq_on_current_page(driver)
    print(f" - 현재 페이지 FAQ 개수: {len(page_data)}")

    for row in page_data:
        row["page"] = page

    all_data.extend(page_data)

    if page < end_page:
        moved = go_to_next_page(driver, page + 1)
        if not moved:
            print(f"{page + 1}페이지로 이동 실패. 중단합니다.")
            break

print("총 수집 개수:", len(all_data))

1페이지 수집 중...
 - 현재 페이지 FAQ 개수: 10
2페이지 수집 중...
 - 현재 페이지 FAQ 개수: 10
3페이지 수집 중...
 - 현재 페이지 FAQ 개수: 10
4페이지 수집 중...
 - 현재 페이지 FAQ 개수: 7
총 수집 개수: 37


In [16]:
for row in all_data[:5]:
    print("PAGE:", row["page"])
    print("Q:", row["question"])
    print("A:", row["answer"])
    print("-" * 80)

PAGE: 1
Q: [민간보조사업] 수소충전소 민간보조 사업 지원기준 및 절차는?
A: A
• 한국자동차환경협회 업무대행업무로 자세한 사항 아래 참조(연료구입비 지원사업 포함)
   - (관련사이트) 한국자동차환경협회(https://www.aea.or.kr/biz/hcInfra.do)
  - (참고자료) 2023년 수소차 보급 및 충전소 설치사업 보조금 업무처리지침(환경부)
  - (기타문의) 070-4027-0584
--------------------------------------------------------------------------------
PAGE: 1
Q: [주요설비] 수소충전소 주요설비 구성은?
A: A
• 튜브트레일러 기준
 1. (수소공급) 튜브트레일러: 수소충전소 내 수소를 공급하는 이동식 저장설비(200bar, 300kg)
2. (수소압축) 압축기: 저장설비의 수소를 고압으로 승압하여 저장용기에 저장(→ 1,000bar)
3. (수소저장) 저장용기: 압축설비에서 고압으로 전환된 수소를 저장(저압, 중압, 고압용기로 구성)
4. (냉각) 냉각기(칠러): 수소 온도 상승에 따른 수소차의 저장탱크 보호를 위해 수소온도를 -40°c까지 하강(압축과 충전시 온도 상승 제어)
5, (충전) 디스펜서: 수소를 수소차량에 충전하는 기기
--------------------------------------------------------------------------------
PAGE: 1
Q: [관계정부부처] 수소충전 구축에 따른 정부기관 역할 분담은?
A: A
• 환경부는 수소충전소 보급 총괄기관이며, 수소 생산·유통과 관련하여 산자부, 국토부 등과 협업하고 있음
  - (환경부) 수소충전소 구축 총괄, 수소충전소 배치계획 수립, 수소충전소 운영비 지원, 수소충전소 인허가 특례 도입 등
 - (산자부)  수소 생산·유통 총괄, 수소충전소 부품 국산화(R&D), 수소연료 생산시설 구축, 수소충전소 안전관리(한국가스안전공사) 등
 - (국토

In [17]:
def go_to_page(driver, page_num):
    try:
        page_link = driver.find_element(
            By.XPATH,
            f"//a[normalize-space()='{page_num}']"
        )

        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", page_link)
        time.sleep(0.5)
        driver.execute_script("arguments[0].click();", page_link)

        time.sleep(2)
        return True

    except:
        return False

In [19]:
all_data = []

start_page = 1
end_page = 4

for page in range(start_page, end_page + 1):
    print(f"{page}페이지 수집 중...")

    page_data = extract_faq_on_current_page(driver)
    print(f" - 현재 페이지 FAQ 개수: {len(page_data)}")

    for row in page_data:
        row["page"] = page

    all_data.extend(page_data)

    if page < end_page:
        moved = go_to_page(driver, page + 1)

        if not moved:
            print(f"{page + 1}페이지 이동 실패. 중단합니다.")
            break

print("총 수집 개수:", len(all_data))

1페이지 수집 중...
 - 현재 페이지 FAQ 개수: 10
2페이지 수집 중...
 - 현재 페이지 FAQ 개수: 10
3페이지 수집 중...
 - 현재 페이지 FAQ 개수: 10
4페이지 수집 중...
 - 현재 페이지 FAQ 개수: 7
총 수집 개수: 37


In [20]:
for row in all_data[:5]:
    print("PAGE:", row["page"])
    print("Q:", row["question"])
    print("A:", row["answer"])
    print("-" * 80)

PAGE: 1
Q: [민간보조사업] 수소충전소 민간보조 사업 지원기준 및 절차는?
A: A
• 한국자동차환경협회 업무대행업무로 자세한 사항 아래 참조(연료구입비 지원사업 포함)
   - (관련사이트) 한국자동차환경협회(https://www.aea.or.kr/biz/hcInfra.do)
  - (참고자료) 2023년 수소차 보급 및 충전소 설치사업 보조금 업무처리지침(환경부)
  - (기타문의) 070-4027-0584
--------------------------------------------------------------------------------
PAGE: 1
Q: [주요설비] 수소충전소 주요설비 구성은?
A: A
• 튜브트레일러 기준
 1. (수소공급) 튜브트레일러: 수소충전소 내 수소를 공급하는 이동식 저장설비(200bar, 300kg)
2. (수소압축) 압축기: 저장설비의 수소를 고압으로 승압하여 저장용기에 저장(→ 1,000bar)
3. (수소저장) 저장용기: 압축설비에서 고압으로 전환된 수소를 저장(저압, 중압, 고압용기로 구성)
4. (냉각) 냉각기(칠러): 수소 온도 상승에 따른 수소차의 저장탱크 보호를 위해 수소온도를 -40°c까지 하강(압축과 충전시 온도 상승 제어)
5, (충전) 디스펜서: 수소를 수소차량에 충전하는 기기
--------------------------------------------------------------------------------
PAGE: 1
Q: [관계정부부처] 수소충전 구축에 따른 정부기관 역할 분담은?
A: A
• 환경부는 수소충전소 보급 총괄기관이며, 수소 생산·유통과 관련하여 산자부, 국토부 등과 협업하고 있음
  - (환경부) 수소충전소 구축 총괄, 수소충전소 배치계획 수립, 수소충전소 운영비 지원, 수소충전소 인허가 특례 도입 등
 - (산자부)  수소 생산·유통 총괄, 수소충전소 부품 국산화(R&D), 수소연료 생산시설 구축, 수소충전소 안전관리(한국가스안전공사) 등
 - (국토

In [21]:
df = pd.DataFrame(all_data)
df = df.drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)

df.to_csv("ev_faq_all_pages.csv", index=False, encoding="utf-8-sig")

print("CSV 저장 완료")
print("최종 데이터 개수:", len(df))

CSV 저장 완료
최종 데이터 개수: 37
